---
title: "Notebook: Where Learning Rates Break"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Reserted/Data-Science-Portfolio/blob/main/blog/learning-rate/learning-rate-colab.ipynb)

This notebook produces every figure and number in the post
[Where Learning Rates Break](learning-rate.md): one hyperparameter, three models (MLP, small CNN, ResNet-18),
Kuzushiji-MNIST.

**How to run.** Open it in Colab → *Runtime → Change runtime type → T4 GPU* → *Runtime → Run all*.
It takes roughly 15 minutes on a T4. The last cell downloads `learning-rate.zip`; unzip it
inside the repo's `blog/` folder so the figures land in `blog/learning-rate/figures/`, where the post
expects them. The zip also holds `results.md` (tables to paste into the post) and `runs.csv`
(every epoch of every run).

## 0. Setup

In [ ]:
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import Markdown, display
from matplotlib.colors import LinearSegmentedColormap
from torch.utils.data import DataLoader
from torchvision import datasets, transforms as T

SLUG = "learning-rate"
OUT = Path(SLUG) / "figures"   # every figure, table and the raw runs land here
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} · torchvision {torchvision.__version__} · "
      f"{torch.cuda.get_device_name() if DEVICE == 'cuda' else 'CPU only: switch the runtime to a T4 GPU first'}")

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "lines.markersize": 6, "font.size": 10, "legend.frameon": False,
    "axes.formatter.min_exponent": 3,   # log axes print 0.5, 2, 10 instead of 5×10⁻¹, 2×10⁰
    "axes.formatter.useoffset": False,  # print 112.02, not "+1.12e2" above the axis
})

## 1. Shared code

Identical in all four notebooks of the series: data loading, the three models, the training loop, and the plots.

In [ ]:
# ---- data ---------------------------------------------------------------------------------
def load(ds):
    """Decode a torchvision dataset once into (images, labels) tensors."""
    xs, ys = zip(*DataLoader(ds, batch_size=1000, num_workers=2))
    return torch.cat(xs), torch.cat(ys)


def holdout(x, y, n, seed=42):
    """Split off n random examples. The seed is fixed, so every run uses the same split."""
    p = torch.randperm(len(x), generator=torch.Generator().manual_seed(seed))
    return (x[p[n:]], y[p[n:]]), (x[p[:n]], y[p[:n]])


def to_device(splits):
    """Normalise every split with the training set's per-channel mean/std, then move it to the GPU once."""
    x = splits["train"][0]
    mean, std = x.mean((0, 2, 3), keepdim=True), x.std((0, 2, 3), keepdim=True)
    out = {k: (((a - mean) / std).to(DEVICE), b.to(DEVICE)) for k, (a, b) in splits.items()}
    for k, (a, b) in out.items():
        print(f"{k:<5} {tuple(a.shape)}")
    return out


def show_samples(x, y, names, path, n=10):
    """One training image from each of the first n classes."""
    fig, axes = plt.subplots(1, n, figsize=(1.4 * n, 1.8))
    for c, ax in enumerate(axes):
        ax.imshow(x[(y == c).nonzero()[0, 0]].permute(1, 2, 0).squeeze(), cmap="gray")
        ax.set_title(names[c], fontsize=8)
        ax.axis("off")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()


# ---- models -------------------------------------------------------------------------------
MODELS = ["MLP", "CNN", "ResNet-18"]


def make_model(name, c, size, n, p=0.0):
    """c×size×size images in, n class scores out. p = dropout rate (0 = no dropout)."""
    if name == "MLP":
        return nn.Sequential(
            nn.Flatten(),
            nn.Linear(c * size * size, 512), nn.ReLU(), nn.Dropout(p),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, n))
    if name == "CNN":
        return nn.Sequential(
            nn.Conv2d(c, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Dropout(p),
            nn.Linear(64 * (size // 4) ** 2, 128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, n))
    m = torchvision.models.resnet18(num_classes=n)          # random init: trained from scratch, no pretrained weights
    m.conv1 = nn.Conv2d(c, 64, 7, 2, 3, bias=False)          # same stem, but accepts c input channels
    m.fc = nn.Sequential(nn.Dropout(p), nn.Linear(512, n))  # ResNet-18 has no dropout of its own: add it before the classifier
    return m


# ---- training -----------------------------------------------------------------------------
@torch.no_grad()
def evaluate(model, x, y, bs=1000):
    """Mean cross-entropy and accuracy in eval mode (dropout off, BatchNorm running statistics)."""
    model.eval()
    loss = correct = 0.0
    for i in range(0, len(x), bs):
        out = model(x[i:i + bs])
        loss += F.cross_entropy(out, y[i:i + bs], reduction="sum").item()
        correct += (out.argmax(1) == y[i:i + bs]).sum().item()
    return loss / len(x), correct / len(x)


def run(model_name, seed, epochs, lr=0.01, batch_size=128, dropout=0.0, weight_decay=0.0):
    """Train one model with SGD (momentum 0.9, constant learning rate). Returns one row of metrics per epoch."""
    hp = dict(lr=lr, batch_size=batch_size, dropout=dropout, weight_decay=weight_decay)
    x, y = DATA["train"]
    torch.manual_seed(seed)   # same seed -> same initial weights for every value of the swept hyperparameter
    model = make_model(model_name, x.shape[1], x.shape[2], int(y.max()) + 1, dropout).to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    order_rng = torch.Generator(device=DEVICE).manual_seed(seed)
    rows, train_time = [], 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        order = torch.randperm(len(x), device=DEVICE, generator=order_rng)
        for i in range(0, len(x) - batch_size + 1, batch_size):   # drops the last partial batch
            idx = order[i:i + batch_size]
            loss = F.cross_entropy(model(x[idx]), y[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        train_time += time.time() - t0
        train_loss, train_acc = evaluate(model, x[:5000], y[:5000])   # fixed training slice, re-scored with dropout off
        val_loss, val_acc = evaluate(model, *DATA["val"])
        weight_norm = torch.cat([p.detach().flatten() for p in model.parameters()]).norm().item()
        rows.append(dict(model=model_name, seed=seed, **hp, epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                         val_loss=val_loss, val_acc=val_acc, weight_norm=weight_norm, seconds=train_time))
    rows[-1]["test_acc"] = evaluate(model, *DATA["test"])[1]   # scored once, after the last epoch
    return rows


def sweep(param, values, epochs, seeds=(0, 1, 2), **fixed):
    """Every value × model × seed. Everything except `param` stays at run()'s defaults (or `fixed`)."""
    x, y = DATA["train"]
    for m in MODELS:
        n = sum(p.numel() for p in make_model(m, x.shape[1], x.shape[2], int(y.max()) + 1).parameters())
        print(f"{m:<10} {n:>12,} parameters")
    rows = []
    for v in values:
        for m in MODELS:
            for s in seeds:
                r = run(m, s, epochs, **fixed, **{param: v})
                rows += r
                print(f"{param} = {v:<7g} {m:<10} seed {s}   val acc {r[-1]['val_acc']:.3f}   train {r[-1]['seconds']:6.1f}s")
    df = pd.DataFrame(rows)
    df.to_csv(OUT / "runs.csv", index=False)
    return df


# ---- figures and tables -------------------------------------------------------------------
COLORS = dict(zip(MODELS, ["#2a78d6", "#eb6834", "#1baf7a"]))   # one fixed colour per model, in every figure
MARKERS = dict(zip(MODELS, ["o", "s", "^"]))
RAMP = LinearSegmentedColormap.from_list("ordinal", ["#86b6ef", "#0d366b"])   # light = small value, dark = large
LABEL = {"lr": "learning rate", "batch_size": "batch size", "dropout": "dropout rate p", "weight_decay": "weight decay λ"}


def final(df):
    return df[df.epoch == df.epoch.max()]


def plot_final(df, param, metric, ylabel, path):
    """Final-epoch metric against the hyperparameter, one line per model. Line = mean over seeds, band = min to max."""
    values = sorted(df[param].unique())
    x = np.arange(len(values))
    last = final(df)
    fig, ax = plt.subplots(figsize=(6.4, 4))
    for m in MODELS:
        g = last[last.model == m].groupby(param)[metric].agg(["mean", "min", "max"]).reindex(values)
        ax.plot(x, g["mean"], marker=MARKERS[m], color=COLORS[m], label=m)
        ax.fill_between(x, g["min"], g["max"], color=COLORS[m], alpha=0.15, linewidth=0)
    ax.set_xticks(x, [f"{v:g}" for v in values])
    ax.set_xlabel(LABEL[param])
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_curves(df, param, metric, ylabel, path, x="epoch", logy=False):
    """One panel per model: metric during training, one line per hyperparameter value (mean over seeds)."""
    values = sorted(df[param].unique())
    fig, axes = plt.subplots(1, len(MODELS), figsize=(13, 3.6))
    for ax, m in zip(axes, MODELS):
        for i, v in enumerate(values):
            d = df[(df.model == m) & (df[param] == v)]
            g = d.groupby("epoch")[[metric, "seconds"]].mean().reset_index()
            ax.plot(g[x], g[metric], color=RAMP(i / max(len(values) - 1, 1)), label=f"{v:g}")
        ax.set_title(m, loc="left")
        ax.set_xlabel("training seconds" if x == "seconds" else x)
        if logy:
            ax.set_yscale("log")
    axes[0].set_ylabel(ylabel)
    axes[-1].legend(title=LABEL[param], loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()


def table(df, param, metric):
    """Final-epoch metric as markdown: one row per value, one column per model, mean ± sd over seeds."""
    t = final(df).groupby([param, "model"])[metric].agg(lambda s: f"{s.mean():.3f} ± {s.std():.3f}").unstack()
    return t[MODELS].to_markdown()


def best(df, param):
    """Each model's best value of `param`, picked on mean validation accuracy, with its test accuracy."""
    means = final(df).groupby(["model", param])[["train_acc", "val_acc", "test_acc"]].mean().reset_index()
    return means.loc[means.groupby("model").val_acc.idxmax()].set_index("model").loc[MODELS].round(3).to_markdown()


def save_report(md):
    (OUT / "results.md").write_text(md, encoding="utf-8")
    display(Markdown(md))

## 2. Data: Kuzushiji-MNIST

`torchvision.datasets.KMNIST`: 70,000 28×28 grayscale images of ten cursive Japanese hiragana
(Kuzushiji), drawn from pre-modern books. Same shape as MNIST, but much harder: the same
character is written in many very different styles. 6,000 training images are held out for
validation.

In [ ]:
tf = T.ToTensor()
train, val = holdout(*load(datasets.KMNIST("data", train=True, download=True, transform=tf)), 6000)
test = load(datasets.KMNIST("data", train=False, download=True, transform=tf))
show_samples(*train, datasets.KMNIST.classes, OUT / "samples.png")
DATA = to_device({"train": train, "val": val, "test": test})

## 3. Sweep

Six learning rates, half a decade apart, × three models × three seeds = 54 runs of 10 epochs.
Batch size 128, no dropout, no weight decay.

In [ ]:
df = sweep("lr", [0.001, 0.003, 0.01, 0.03, 0.1, 0.3], epochs=10)

## 4. Figures

In [ ]:
plot_final(df, "lr", "val_acc", "final validation accuracy", OUT / "accuracy-vs-lr.png")
plot_curves(df, "lr", "train_loss", "training loss", OUT / "train-loss-curves.png", logy=True)

## 5. Tables

In [ ]:
save_report(f"""### Final validation accuracy (mean ± sd, 3 seeds)

{table(df, "lr", "val_acc")}

### Best learning rate per model (picked on validation, then scored on test)

{best(df, "lr")}
""")

## 6. Download

In [ ]:
archive = shutil.make_archive(SLUG, "zip", root_dir=".", base_dir=SLUG)   # zip holds <slug>/figures/...
print("Unzip into the repo's blog/ folder:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass   # not on Colab: the zip is next to this notebook